# Feature Experiments (Appendix)

This notebook documents experiments that did NOT improve model performance.
Included for transparency and to demonstrate reasoning about feature engineering decisions.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder

DATA_PROCESSED = Path("../data/processed")

## Experiment 1: Citation severity categorization

**Hypothesis**: Categorizing citations by type (sterility, process validation,
quality system, etc.) would give the model more signal than just counting total citations.

**Approach**: Mapped each CFR code to a severity category and created per-category counts.

**Result**: No improvement. The model already captured the signal via `total_prior_citations`
and `max_citations_single_inspection`. Desglosing by category adds granularity but
any high citation count (regardless of type) already predicts OAI.

## Experiment 2: Warning letter subject type features

**Hypothesis**: A CGMP warning letter is a stronger OAI predictor than an FSVP letter.

**Approach**: Extracted subject categories (cgmp, adulterated, misbranded, etc.) from
warning letter subjects and created per-category counts.

**Result**: No improvement. The features are too sparse — only 0.02-0.24% of inspections
have any non-zero value. Not enough signal for the model to learn from.

| Feature | Non-zero rows | % of dataset |
|---|---|---|
| wl_cgmp | 312 | 0.19% |
| wl_adulterated | 402 | 0.24% |
| wl_unapproved_drug | 76 | 0.05% |
| wl_misbranded | 121 | 0.07% |
| wl_medical_device | 74 | 0.04% |
| wl_compounding | 29 | 0.02% |
| wl_fsvp | 29 | 0.02% |

## Experiment 3: Removing the dominant feature (`last_classification_oai`)

**Hypothesis**: The model is over-reliant on `last_classification_oai` (50-72% importance).
Without it, performance should drop significantly.

**Result**: Performance is virtually unchanged. This is because other features
encode the same information (`n_prior_oai`, `recent_oai_rate`, `trend_worsening`).
When the direct feature is removed, the model redistributes to alternatives.

This is an important finding for production robustness: the model doesn't collapse
if any single feature becomes unavailable.

In [2]:
# Reproduce the experiment
df = pd.read_parquet(DATA_PROCESSED / "features.parquet")
df["year"] = df["inspection_date"].dt.year

train = df[df["year"] < 2025].copy()
test = df[df["year"] >= 2025].copy()

CATEGORICAL = ["product_type", "country", "project_area"]
DROP = ["inspection_id", "fei_number", "inspection_date", "target_oai", "year"]
REDUNDANT = ["has_warning_letter", "has_recall", "has_published_483", "pct_oai", "pct_vai"]

NUMERIC = [c for c in df.columns if c not in DROP + CATEGORICAL + REDUNDANT]
for col in CATEGORICAL:
    le = LabelEncoder()
    le.fit(df[col].astype(str))
    train[col + "_enc"] = le.transform(train[col].astype(str))
    test[col + "_enc"] = le.transform(test[col].astype(str))

ENCODED = [c + "_enc" for c in CATEGORICAL]
FEATURES_ALL = NUMERIC + ENCODED
FEATURES_NO_LAST = [f for f in FEATURES_ALL if f != "last_classification_oai"]

y_train = train["target_oai"].values
y_test = test["target_oai"].values
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

results = {}

# With all features
X_train = train[FEATURES_ALL].values
X_test = test[FEATURES_ALL].values
model_all = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=n_neg / n_pos, random_state=42,
    eval_metric="aucpr", early_stopping_rounds=20,
)
model_all.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
y_proba = model_all.predict_proba(X_test)[:, 1]
results["All features"] = {
    "ROC-AUC": roc_auc_score(y_test, y_proba),
    "PR-AUC": average_precision_score(y_test, y_proba),
}

# Without last_classification_oai
X_train_nl = train[FEATURES_NO_LAST].values
X_test_nl = test[FEATURES_NO_LAST].values
model_no_last = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=n_neg / n_pos, random_state=42,
    eval_metric="aucpr", early_stopping_rounds=20,
)
model_no_last.fit(X_train_nl, y_train, eval_set=[(X_test_nl, y_test)], verbose=False)
y_proba_nl = model_no_last.predict_proba(X_test_nl)[:, 1]
results["Without last_classification_oai"] = {
    "ROC-AUC": roc_auc_score(y_test, y_proba_nl),
    "PR-AUC": average_precision_score(y_test, y_proba_nl),
}

print(f"{'Model':<40} {'ROC-AUC':>8} {'PR-AUC':>8}")
print("-" * 60)
for name, metrics in results.items():
    print(f"{name:<40} {metrics['ROC-AUC']:>8.4f} {metrics['PR-AUC']:>8.4f}")

Model                                     ROC-AUC   PR-AUC
------------------------------------------------------------
All features                               0.8714   0.3606
Without last_classification_oai            0.8753   0.3755


In [3]:
# Feature importance WITHOUT last_classification_oai
importances = pd.Series(model_no_last.feature_importances_, index=FEATURES_NO_LAST)
print("\nTop 15 features (without last_classification_oai):")
print(importances.sort_values(ascending=False).head(15).to_string())


Top 15 features (without last_classification_oai):
n_prior_oai                        0.446341
n_prior_nai                        0.107604
project_area_enc                   0.062422
product_type_enc                   0.060427
last_classification_vai            0.053601
recent_oai_rate                    0.047070
n_prior_inspections                0.030695
avg_citations_per_inspection       0.023382
days_since_last_inspection         0.022640
country_enc                        0.022435
max_citations_single_inspection    0.021988
n_recalls                          0.015247
n_class_I_recalls                  0.012530
trend_worsening                    0.012121
n_published_483s                   0.011862


## Conclusions

1. **Citation categorization and WL type features don't improve metrics** — the raw counts
   already capture the available signal. The categories are either too sparse (WL) or
   redundant with the total (citations).

2. **`last_classification_oai` is not a single point of failure** — removing it doesn't
   hurt performance because `n_prior_oai`, `recent_oai_rate`, and `trend_worsening`
   contain equivalent information. Feature importance ≠ information uniqueness.

3. **ROC-AUC ~0.87 is likely the ceiling with public data** — to improve further,
   we would need Qualifyze's private audit data, which provides signal that public
   FDA records cannot (e.g., internal quality observations before they become citations).